# Zachary Weed


In [9]:
pip install flash-attn --no-build-isolation -v

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


# Head Class

In [16]:
import torch
import torch.nn as nn

from torch.nn import functional as F
from flash_attn import flash_attn_func

class Head(nn.Module):
  def __init__(self, head_size, attention_type, number_of_embeddings = 32, block_size = 8):
    super().__init__()

    # compute projections from embeddings
    # into a different space, as:
    # Query, Key, and Value
    self.query = nn.Linear(number_of_embeddings, head_size, bias=False)
    self.key = nn.Linear(number_of_embeddings, head_size, bias = False)
    self.value = nn.Linear(number_of_embeddings, head_size, bias = False)
    self.attention_type = attention_type

    # registers causal mask on  model
    self.register_buffer('tril', self.build_causal_mask(block_size))

  # Ensures attention is
  # only maintained for tokens
  # that have already been processed.
  # The tensor is essentially like this:
  #   a | b | c
  # a 1 | 0 | 0
  # b 1 | 1 | 0
  # c 1 | 1 | 1
  # Each row represents what
  # that token can attend with, so:
  #   1) 'a' can attend with a, but not b & c,
  #   2) 'b' can attend with a, and b, but not c, &
  #   3) 'c' can attend with all
  #
  # Step-by-step
  #
  # With respect to time, if at step one, we are processing
  # row number 1, and thus token 'a', then:
  #   a | b | c
  # a 1 | 0 | 0 present
  # b 1 | 1 | 0 future
  # c 1 | 1 | 1 future
  #
  # Then as it shifts to token 'b', then:
  #   a | b | c
  # a 1 | 0 | 0 past
  # b 1 | 1 | 0 present
  # c 1 | 1 | 1 future
  #
  # Then as it shifts to token 'c':
  #   a | b | c
  # a 1 | 0 | 0 past
  # b 1 | 1 | 0 past
  # c 1 | 1 | 1 present
  #
  # Thus, if we are at token 'a' in step one,
  # then 'a' can map with its self
  # and no other tokens. And if we are at
  # token 'b' in step two, then it can map with
  # a 'past token' ('a'), and its self
  #
  def build_causal_mask(self, block_size):
    square_matrix = torch.ones(block_size, block_size)
    triangular_matrix = torch.tril(square_matrix)
    return triangular_matrix


  def forward(self, x):
    if self.attention_type == "traditional_attention_forward":
      return self.traditional_attention_forward(x)
    else:
      return self.flash_attention_forward(x)

  def traditional_attention_forward(self, x):
    batch, time, channels = x.shape

    # matrix multiplication of keys and queries
    # representing what is contained, and what
    # is being looked for, respectively.
    q = self.query(x)
    k = self.key(x)

    attention_score_matrix = q @ k.transpose(-2, -1) * channels ** -0.5

    # since we have the triangular matrix, as a causal mask,
    # here we map the mask so that anything with value of 0
    # (not allowed) becomes mapped as True, and anything
    # with a value of 1 becomes mapped as False.
    # The -inf is used as a means of swapping
    # True with -inf so the softmax() function provides 0
    # probability when applying the causal mask
    #
    # Thus, if we have
    #   a | b | c |
    # a 1 | 0 | 0 |
    # b 1 | 1 | 0 |
    # c 1 | 1 | 1 |
    #
    # Then it becomes the vector below,
    # before transforming with -inf
    # (with F means allowed and T means disallowed)
    #
    #   a | b | c |
    # a F | T | T |
    # b F | F | T |
    # c F | F | F |
    #
    # Transforming with -inf produces
    #   a | b    | c    |
    # a F | -inf | -inf |
    # b F | F    | -inf |
    # c F | F    | F    |
    #
    attention_score_matrix = attention_score_matrix.masked_fill(
      self.tril[:time, :time] == 0, float('-inf')
    )

    # computes aforementioned softmax, now that
    # values are properly standardized
    # The softmax makes sure that anything
    # with -inf has a 0 probability
    #   a | b | c |
    # a F | 0 | 0 |
    # b F | F | 0 |
    # c F | F | F |
    attention_score_matrix = F.softmax(attention_score_matrix, dim = -1)
    return attention_score_matrix @ self.value(x)

  # Essentially identical in implementation with the original Flashattention function
  def flash_attention_forward(self, x):
    q = self.query(x)
    k = self.key(x)
    v = self.value(x)

    # so we can provide another dimension
    q = q.unsqueeze(2)
    k = k.unsqueeze(2)
    v = v.unsqueeze(2)

    # removes the additional dimension on output so we can verify similarity
    return flash_attn_func(q, k, v, causal = True).squeeze(2)

# Multihead Class

In [17]:
class MultiHeadAttention(nn.Module):
  def __init__(self, number_of_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList(
      [
        Head(head_size) for _ in range(number_of_heads)
      ]
    )

  def forward(self, x):
    return torch.cat([head(x) for head in self.heads], dim = -1)

# CUDA Benchmarking Cell

In [22]:
def benchmark(fn, x, warmup = 20, iterations = 100):
  for _ in range(warmup):
    _ = fn(x)
  torch.cuda.synchronize()

  start = torch.cuda.Event(enable_timing = True)
  end = torch.cuda.Event(enable_timing = True)

  start.record()

  for _ in range(iterations):
    _ = fn(x)

  end.record()
  torch.cuda.synchronize()

  total_milliseconds = start.elapsed_time(end)

  return total_milliseconds / iterations



# Calling Traditional and Flashattention and Verifying Similarity

In [25]:
device = "cuda"
data_type = torch.float16

batch_size = 2
time = 8
channels = 32
head_size = 32

# Builds head
head = Head(
  head_size = head_size,
  attention_type = "traditional_attention_forward",
  number_of_embeddings = channels,
  block_size = time
).to(device).to(data_type)

# Builds X for testing
x = torch.randn(batch_size, time, channels, device = device, dtype = data_type)

head.eval()

with torch.no_grad():
  # Computes with traditional architecture
  traditional = benchmark(head.traditional_attention_forward, x)
  # Computes with flash architecture
  flash = benchmark(head.flash_attention_forward, x)


# Identifies that the Results are essentially equivalent
print(f"Traditional: {traditional}")
print(f"Flash: {flash}")
print(f"How Much Faster: {traditional/flash:.2f}x")



Traditional: 0.29715456008911134
Flash: 0.26838016510009766
How Much Faster: 1.11x
